In [1]:
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

import MDAnalysis as mda
from MDAnalysis.analysis import distances
import networkx as nx

from polym_analysis import *
from gromacs_topology import *

/Users/nate_schwindt/anaconda3/envs/mda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/nate_schwindt/anaconda3/envs/mda/lib/python3.10/site-packages/Bio/Application/__init__.py:40: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


In [2]:
def random_amides(polym, num, seed=12345):
    '''Randomly select num of amide bonds to break'''

    sel = polym.universe.select_atoms('type n')
    n = len(sel)
    rng = np.random.default_rng(seed=seed)
    rand_idx = rng.choice(np.arange(0,n), size=num, replace=False)
    
    return sel[rand_idx]


def get_bond(top, atomtype1, atomtype2):
    '''Get the bond between two atom types in a topology.'''

    atomtypes = [atomtype1, atomtype2]

    for b in top.bonds:
        my_types = [a.type for a in b.atoms]
        if Counter(my_types) == Counter(atomtypes):
            return b
    return None


def get_angle(top, atomtype1, atomtype2, atomtype3):
    '''Get the angle among three atom types in a topology.'''

    atomtypes = [atomtype1, atomtype2, atomtype3]

    for a in top.angles:
        my_types = [a.type for a in a.atoms]
        if Counter(my_types) == Counter(atomtypes):
            return a
    return None


def get_dihedral(top, atomtype1, atomtype2, atomtype3, atomtype4, is_improper=False):
    '''Get the dihedral among four atom types in a topology. Should specify if this dihedral is improper.'''

    atomtypes = [atomtype1, atomtype2, atomtype3, atomtype4]
    improper_types = [2, 4] # either harmonic (2) or periodic (4) improper

    for d in top.dihedrals:
        my_types = [a.type for a in d.atoms]
        if Counter(my_types) == Counter(atomtypes):
            if is_improper and d.type in improper_types:
                return d
            
            if not is_improper and d.type not in improper_types:
                return d
            
    return None

## Get the Cl and OH locations

In [3]:
path = '/Users/nate_schwindt/Projects/polymer_membrane/polyamide_XAS_MD/Na/NaCl_polymer/50pct_deprotonated/OPC3/placed_Na/'

xlink_c = 'c and bonded type n'
xlink_n = 'n'
term_n = 'nh'
cl_type = 'cl'
oh_type = 'oh'
ow_type = 'OW'
hw_type = 'HW'
cation_type = 'Na'
anion_type = 'Cl'

poly = PolymAnalysis(path+'prod.gro', frmt='GRO', tpr_file=path+'prod.tpr',
                    xlink_c=xlink_c, xlink_n=xlink_n, term_n=term_n,
                    cl_type=cl_type, oh_type=oh_type, ow_type=ow_type, hw_type=hw_type,
                    cation_type=cation_type, anion_type=anion_type)

Initializing polyamide membrane for analysis...
PA membrane consists of 716 MPD monomers, 444 TMC monomers, 6120 waters for a total of 36086 atoms
Finished loading PA membrane for analysis!



In [4]:
# determine waters to remove and positions to add Cl and OH
target_crosslink_pct = 0.8
radius_for_water_search = 6  # angstroms
rng_seed = 3

n_xlink = len(poly.universe.select_atoms('type n'))
n_possible = len(poly.universe.select_atoms('type n nh'))
n_to_break = n_xlink - int(target_crosslink_pct * n_possible)

print(f'Going from {100 * n_xlink / n_possible:.1f}% crosslinking to {100 * (n_xlink - n_to_break) / n_possible:.1f}% crosslinking by breaking {n_to_break} amide bonds')                                  
print('Number of water molecules before breaking crosslinks:', len(poly.universe.select_atoms('resname SOL').residues))

# delete amide bonds
amide_Ns = random_amides(poly, n_to_break, seed=rng_seed)
amide_Cs = poly.universe.select_atoms('(type c) and (bonded group amide_Ns)', amide_Ns=amide_Ns)

to_delete = []

print(f'Trying to delete {n_to_break} waters and adding {n_to_break} Cl atoms')
Cl_positions = np.zeros((n_to_break, 3))
for i,n in enumerate(amide_Ns):
    my_waters = poly.universe.atoms.select_atoms(f'(type OW) and (sphzone {radius_for_water_search} index {n.index})')
    if len(my_waters) == 0:
        raise ValueError('No waters found near amide N atom index', n.index)
    
    # save the positions of the waters to be replaced
    Cl_positions[i] = my_waters[0].position

    # remove the water molecule
    to_delete.append(my_waters[0].residue)
    
# locate nearby waters to replace with OH
print(f'Trying to delete {n_to_break} waters and adding {n_to_break} O atoms and {n_to_break} H atoms')
O_positions = np.zeros((n_to_break, 3))
H_positions = np.zeros((int(n_to_break/2), 3))
for i,c in enumerate(amide_Cs):
    my_waters = poly.universe.atoms.select_atoms(f'(type OW) and (sphzone {radius_for_water_search} index {c.index})')
    if len(my_waters) == 0:
        raise ValueError('No waters found near amide C atom index', c.index)
    
    # save the positions of the waters to be replaced
    O_positions[i] = my_waters[0].position

    if i < int(n_to_break/2):
        H_positions[i] = my_waters[0].bonded_atoms[0].position  # for 50% deprotonated, we want to add O- for half and OH for half

    # remove the water molecule
    to_delete.append(my_waters[0].residue)

to_delete = list(set(to_delete))  # remove duplicates

# Now delete all waters at once
print(f'Actually deleting {len(to_delete)} water molecules')
for i,water_res in enumerate(to_delete):
    if i == 0:
        all_waters_to_delete = water_res.atoms
    else:
        all_waters_to_delete += water_res.atoms

to_delete_idx = all_waters_to_delete.indices
poly.universe.delete_bonds([b for b in all_waters_to_delete.bonds]) # also all bonds associated with deleted waters
poly.universe.atoms = poly.universe.atoms.subtract(all_waters_to_delete)

# reassign residue numbers after deleted waters
dims = poly.universe.dimensions # save original dimensions
n_residues = len(poly.atoms.residues)
resids = np.arange(1, n_residues+1)
poly.universe.add_TopologyAttr('resid', list(resids))

print('Number of water molecules after breaking crosslinks:', len(poly.universe.select_atoms('resname SOL').residues))
print(f'Found {len(Cl_positions)} positions for Cl atoms to be added')
print(f'Found {len(O_positions)} positions for O atoms to be added')
print(f'Found {len(H_positions)} positions for H atoms to be added')

Going from 93.2% crosslinking to 80.0% crosslinking by breaking 189 amide bonds
Number of water molecules before breaking crosslinks: 6120
Trying to delete 189 waters and adding 189 Cl atoms
Trying to delete 189 waters and adding 189 O atoms and 189 H atoms
Actually deleting 206 water molecules
Number of water molecules after breaking crosslinks: 5914
Found 189 positions for Cl atoms to be added
Found 189 positions for O atoms to be added
Found 94 positions for H atoms to be added


## Get the new parameters from the fragment parameterization

In [5]:
# Get bond parameters for new bonds from the parameter topology
param_gro = PolymAnalysis('2MPD_2TMC_1P_Cl_box.gro', frmt='GRO', tpr_file='2MPD_2TMC_1P_Cl_converted.tpr', 
                            xlink_c=xlink_c, xlink_n=xlink_n, term_n=term_n, cl_type=cl_type, 
                            ow_type=ow_type, hw_type=hw_type)
param_top = GromacsTopology('2MPD_2TMC_1P_Cl_converted.top')

cl = param_gro.universe.select_atoms('type cl')
cl_atom = param_top.atoms[cl.indices[0]]
term_c = param_gro.universe.select_atoms('(type c) and (not bonded type n)')
o = param_gro.universe.select_atoms('(type o) and (bonded group term_c)', term_c=term_c)
o_atom = param_top.atoms[o.indices[0]]
oh = param_gro.universe.select_atoms('type oh')
oh_atom = param_top.atoms[oh.indices[0]]
ho = param_gro.universe.select_atoms('type ho')
ho_atom = param_top.atoms[ho.indices[0]]
nh_type = param_top.atomtypes['nh'] # broken amide N changes to type nh (amine N)

# bond parameters
nh_cl_bond = get_bond(param_top, 'nh', 'cl')
c_oh_bond = get_bond(param_top, 'c', 'oh')
oh_ho_bond = get_bond(param_top, 'oh', 'ho')
c_o_bond = get_bond(param_top, 'c', 'o')

print(f'\nnh-cl bond parameters: {[p for p in nh_cl_bond._params]}')
print(f'c-oh bond parameters: {[p for p in c_oh_bond._params]}')
print(f'oh-ho bond parameters: {[p for p in oh_ho_bond._params]}')
print(f'c-o bond parameters: {[p for p in c_o_bond._params]}')

# angle parameters
ca_nh_cl_angle = get_angle(param_top, 'ca', 'nh', 'cl')
hn_nh_cl_angle = get_angle(param_top, 'hn', 'nh', 'cl')
ca_c_oh_angle = get_angle(param_top, 'ca', 'c', 'oh')
c_oh_ho_angle = get_angle(param_top, 'c', 'oh', 'ho')
o_c_oh_angle = get_angle(param_top, 'o', 'c', 'oh')
o_c_o_angle = get_angle(param_top, 'o', 'c', 'o')
ca_c_o_angle = get_angle(param_top, 'ca', 'c', 'o')

print(f'\nca-nh-cl angle parameters: {[p for p in ca_nh_cl_angle._params]}')
print(f'hn-nh-cl angle parameters: {[p for p in hn_nh_cl_angle._params]}')
print(f'ca-c-oh angle parameters: {[p for p in ca_c_oh_angle._params]}')
print(f'c-oh-ho angle parameters: {[p for p in c_oh_ho_angle._params]}')
print(f'o-c-oh angle parameters: {[p for p in o_c_oh_angle._params]}')
print(f'o-c-o angle parameters: {[p for p in o_c_o_angle._params]}')

# dihedral parameters
# ca_nh_cl_hn_imp = get_dihedral(param_top, 'ca', 'nh', 'cl', 'hn', is_improper=True) # this dihedral does not actually appear in the moltemplate GAFF topology
ca_ca_nh_cl_dih = get_dihedral(param_top, 'ca', 'ca', 'nh', 'cl')
ca_c_o_oh_imp = get_dihedral(param_top, 'ca', 'c', 'o', 'oh', is_improper=True)
o_c_oh_ho_dih = get_dihedral(param_top, 'o', 'c', 'oh', 'ho')
ca_c_oh_ho_dih = get_dihedral(param_top, 'ca', 'c', 'oh', 'ho')
ca_ca_c_oh_dih = get_dihedral(param_top, 'ca', 'ca', 'c', 'oh')
ca_c_o_o_imp = get_dihedral(param_top, 'ca', 'c', 'o', 'o', is_improper=True)
ca_ca_c_o_dih = get_dihedral(param_top, 'ca', 'ca', 'c', 'o')

# print(f'\nca-nh-cl-hn improper parameters: {[p for p in ca_nh_cl_hn_imp._params]}')
print(f'\nca-ca-nh-cl dihedral parameters: {[p for p in ca_ca_nh_cl_dih._params]}')
print(f'ca-c-o-oh improper parameters: {[p for p in ca_c_o_oh_imp._params]}')
print(f'o-c-oh-ho dihedral parameters: {[p for p in o_c_oh_ho_dih._params]}')
print(f'ca-c-oh-ho dihedral parameters: {[p for p in ca_c_oh_ho_dih._params]}')
print(f'ca-ca-c-oh dihedral parameters: {[p for p in ca_ca_c_oh_dih._params]}')
print(f'ca-c-o-o improper parameters: {[p for p in ca_c_o_o_imp._params]}')
print(f'ca-ca-c-o dihedral parameters: {[p for p in ca_ca_c_o_dih._params]}')

Initializing polyamide membrane for analysis...
PA membrane consists of 2 MPD monomers, 1 TMC monomers, 0 waters for a total of 63 atoms
Finished loading PA membrane for analysis!



1 molecule types in the topology for a total of 1 molecules

nh-cl bond parameters: [0.1763, 253717.76]
c-oh bond parameters: [0.1306, 390283.52]
oh-ho bond parameters: [0.0974, 309281.28]
c-o bond parameters: [0.1214, 542246.4]

ca-nh-cl angle parameters: [113.15, 482.58256]
hn-nh-cl angle parameters: [104.14, 335.30576]
ca-c-oh angle parameters: [113.44, 586.68048]
c-oh-ho angle parameters: [107.37, 428.35792]
o-c-oh angle parameters: [122.88, 647.51584]
o-c-o angle parameters: [130.38, 654.12656]

ca-ca-nh-cl dihedral parameters: [180.0, 4.3932, 2.0]
ca-c-o-oh improper parameters: [-1.0, 43.932, 2.0]
o-c-oh-ho dihedral parameters: [15.8992, 19.2464, 0.0, 0.0]
ca-c-oh-ho dihedral parameters: [180.0, 9.6232, 2.0]
ca-ca-c-oh dihedral parameters: [180.0, 4.184, 2.0]
ca-c-o-o improper parameters: [-1.0, 43.9

## Add atoms, bonds, angles, dihedrals

In [6]:
path = '/Users/nate_schwindt/Projects/polymer_membrane/polyamide-REACTER/Lx5_Ly5_Lz10/chlorination/50pct_deprotonated/NaCl_0.2M/hydrolyzed_chlorination/random_10pct/'

xlink_c = 'c and bonded type n'
xlink_n = 'n'
term_n = 'nh'
cl_type = 'cl'
oh_type = 'oh'
ow_type = 'OW'
hw_type = 'HW'
cation_type = 'Na'
anion_type = 'Cl'

poly = PolymAnalysis(path+'broken_bonds_sorted.gro', frmt='GRO', tpr_file=path+'broken_bonds.tpr',
                    xlink_c=xlink_c, xlink_n=xlink_n, term_n=term_n,
                    cl_type=cl_type, oh_type=oh_type, ow_type=ow_type, hw_type=hw_type,
                    cation_type=cation_type, anion_type=anion_type)

top = GromacsTopology('./broken_bonds.top', verbose=False)

Initializing polyamide membrane for analysis...
PA membrane consists of 716 MPD monomers, 381 TMC monomers, 5914 waters for a total of 35468 atoms
Finished loading PA membrane for analysis!



121 molecule types in the topology for a total of 6120 molecules


In [7]:
# add new atoms to GRO file
broken_amide_Ns = []
for orig_N in amide_Ns: # build atomgroup by matching the position of the amide N in the original polymer to the broken amide N in the updated polymer
    pos = orig_N.position
    my_Ns = poly.universe.select_atoms(f'(type n) and (not bonded type c) and (point {pos[0]} {pos[1]} {pos[2]} 0.01)')
    if len(my_Ns) != 1:
        raise ValueError(f'{len(my_Ns)} amide Ns found near original amide N position', pos)
    my_N = my_Ns[0]
    broken_amide_Ns.append(my_N)

broken_amide_Ns = mda.AtomGroup(broken_amide_Ns)

broken_amide_Cs = []
for orig_C in amide_Cs: # build atomgroup by matching the position of the amide C in the original polymer to the broken amide C in the updated polymer
    pos = orig_C.position
    my_Cs = poly.universe.select_atoms(f'(type c) and (not bonded type n) and (point {pos[0]} {pos[1]} {pos[2]} 0.01)')
    if len(my_Cs) != 1:
        raise ValueError(f'{len(my_Cs)} broken amide Cs found near original amide C position', pos)
    my_C = my_Cs[0]
    broken_amide_Cs.append(my_C)

broken_amide_Cs = mda.AtomGroup(broken_amide_Cs)
print(f'Adding {len(broken_amide_Ns)} Cl atoms and {len(broken_amide_Cs)} O atoms')

Adding 189 Cl atoms and 189 O atoms


In [8]:
# Create empty universe for new atoms
new_atom_resindices = np.concatenate((
    broken_amide_Ns.resindices,
    broken_amide_Cs.resindices,
    broken_amide_Cs.resindices[:len(H_positions)]  # only half the O atoms get H atoms
))

n_added = len(Cl_positions) + len(O_positions) + len(H_positions)
new_atoms_u = mda.Universe.empty(
    n_atoms=n_added,
    n_residues=len(poly.universe.residues),  # Don't create new residues, but use the existing residues
    atom_resindex=new_atom_resindices,
    trajectory=True
)

# Set atom attributes
n_cl = len(Cl_positions)
n_o = len(O_positions)
n_h = len(H_positions)

# current stacked order: [Cl (0..n_cl-1), O (n_cl..n_cl+n_o-1), H (n_cl+n_o..)]
# base_O = n_cl + np.arange(n_o)
# base_H = n_cl + n_o + np.arange(n_h)

# # build alternating O,H of length 2*n_h
# alternating = np.empty(2 * n_h, dtype=int)
# alternating[0::2] = base_O[:n_h]   # O indices for even slots
# alternating[1::2] = base_H         # H indices for odd slots

# # remaining O's (after using first n_h for alternating)
# remaining_O = base_O[n_h:]

# # final index order: Cl, alternating (O/H), remaining O
# idx = np.concatenate([np.arange(n_cl, dtype=int), alternating, remaining_O])

# apply to positions and names
positions = np.vstack([Cl_positions, O_positions, H_positions])   # shape (n_cl+n_o+n_h,3)
# new_positions = positions[idx]

names = np.array(['Cl'] * n_cl + ['O'] * n_o + ['H'] * n_h)
# new_names = names[idx]

# new_atoms_u.add_TopologyAttr('name', new_names)
# new_atoms_u.add_TopologyAttr('type', new_names) # does not matter that these are not correct atomtypes
# new_atoms_u.add_TopologyAttr('element', new_names)
# new_atoms_u.add_TopologyAttr('resname', [res.resname for res in poly.universe.residues])
# new_atoms_u.add_TopologyAttr('resid', [res.resid for res in poly.universe.residues])

new_atoms_u.add_TopologyAttr('name', names)
new_atoms_u.add_TopologyAttr('type', names) # does not matter that these are not correct atomtypes
new_atoms_u.add_TopologyAttr('element', names)
new_atoms_u.add_TopologyAttr('resname', [res.resname for res in poly.universe.residues])
new_atoms_u.add_TopologyAttr('resid', [res.resid for res in poly.universe.residues])

# Set positions
# new_atoms_u.atoms.positions = new_positions
new_atoms_u.atoms.positions = positions

# Merge with existing universe
merged_u = mda.Merge(poly.universe.atoms, new_atoms_u.atoms)
merged_u.dimensions = poly.universe.dimensions

print(f'Original number of atoms: {len(poly.universe.atoms)}')
print(f'Added {n_added} new atoms to existing residues')
print(f'Total atoms: {len(merged_u.atoms)}')

merged_u.atoms.write('./added_atoms.gro')

Original number of atoms: 35468
Added 472 new atoms to existing residues
Total atoms: 35940


In [9]:
# add Cl to the N on broken amide bonds
total_atoms = len(poly.universe.atoms)
for my_N in broken_amide_Ns:
    my_N_top = top.atoms[my_N.index]

    # pull most attributes from the parameterized fragment with Cl
    atype = cl_atom.type
    atomname = cl_atom.atomname
    cgnr = cl_atom.cgnr
    charge = cl_atom.charge
    mass = cl_atom.mass
    atomtype = cl_atom.atomtype

    # pull other attributes from the current molecule
    num = my_N_top.molecule.n_atoms + 1
    resnum = my_N_top.resnum
    resname = my_N_top.resname
    molecule = my_N_top.molecule

    # create new atom and add to molecule
    new_atom_line = f'{num:7d} {atype:4s}\t\t{resnum} {resname:8s} {atomname:9s}\t{cgnr:d}\t{charge:.8f}\t{mass:.8f}\n'
    new_atom = Atom(total_atoms, new_atom_line)
    new_atom.atomtype = atomtype
    new_atom.molecule = molecule
    molecule.atoms.append(new_atom)

    # change N atom type from 'n' to 'nh'
    my_N_top.atomtype = nh_type
    my_N_top.type = 'nh'

    # add new bonds, angles, dihedrals to the molecule
    my_nh_cl_bond = Bond(new_atom, my_N_top, nh_cl_bond.type, nh_cl_bond._params)
    molecule.bonds.append(my_nh_cl_bond)

    hn_on_N = [a for a in my_N.bonded_atoms if a.type == 'hn'][0]
    hn_top = top.atoms[hn_on_N.index]
    my_hn_nh_cl_angle = Angle(hn_top, my_N_top, new_atom, hn_nh_cl_angle.type, hn_nh_cl_angle._params)
    molecule.angles.append(my_hn_nh_cl_angle)

    ca_on_N = [a for a in my_N.bonded_atoms if a.type == 'ca'][0]
    ca_top = top.atoms[ca_on_N.index]
    my_ca_nh_cl_angle = Angle(ca_top, my_N_top, new_atom, ca_nh_cl_angle.type, ca_nh_cl_angle._params)
    molecule.angles.append(my_ca_nh_cl_angle)

    for ang in my_N_top.angles:
        a1, a2, a3 = ang.atoms
        my_types = [a.type for a in ang.atoms]

        if Counter(my_types) == Counter(['ca', 'nh', 'hn']):
            continue  # this improper is skipped in the moltemplate GAFF topology
        
        dih = get_dihedral(param_top, a1.type, a2.type, a3.type, new_atom.type)
        my_dihedral = Dihedral(a1, a2, a3, new_atom, dih.type, dih._params)
        molecule.dihedrals.append(my_dihedral)

    print(f'Added Cl atom to broken amide N atom index {my_N.index} as atom index {total_atoms}')
    total_atoms += 1

Added Cl atom to broken amide N atom index 6827 as atom index 35468
Added Cl atom to broken amide N atom index 8859 as atom index 35469
Added Cl atom to broken amide N atom index 5726 as atom index 35470
Added Cl atom to broken amide N atom index 15983 as atom index 35471
Added Cl atom to broken amide N atom index 9242 as atom index 35472
Added Cl atom to broken amide N atom index 8808 as atom index 35473
Added Cl atom to broken amide N atom index 5837 as atom index 35474
Added Cl atom to broken amide N atom index 7920 as atom index 35475
Added Cl atom to broken amide N atom index 9977 as atom index 35476
Added Cl atom to broken amide N atom index 9390 as atom index 35477
Added Cl atom to broken amide N atom index 13989 as atom index 35478
Added Cl atom to broken amide N atom index 11925 as atom index 35479
Added Cl atom to broken amide N atom index 14001 as atom index 35480
Added Cl atom to broken amide N atom index 9405 as atom index 35481
Added Cl atom to broken amide N atom index 8

In [ ]:
# add O to the C on broken amide bonds
n_protonated = len(H_positions)
for n,my_C in enumerate(broken_amide_Cs):
    my_C_top = top.atoms[my_C.index]

    # add O atom
    if n < n_protonated:
        ref_atom = oh_atom # 50% should be OH
    else:
        ref_atom = o_atom

    # pull most attributes from the parameterized fragment
    atype = ref_atom.type
    atomname = ref_atom.atomname
    cgnr = ref_atom.cgnr
    charge = ref_atom.charge
    mass = ref_atom.mass
    atomtype = ref_atom.atomtype

    # pull other attributes from the current molecule
    num = my_C_top.molecule.n_atoms + 1
    resnum = my_C_top.resnum
    resname = my_C_top.resname
    molecule = my_C_top.molecule

    # create new atom and add to molecule
    new_atom_line = f'{num:7d} {atype:4s}\t\t{resnum} {resname:8s} {atomname:9s}\t{cgnr:d}\t{charge:.8f}\t{mass:.8f}\n'
    new_atom = Atom(total_atoms, new_atom_line)
    new_atom.atomtype = atomtype
    new_atom.molecule = molecule
    molecule.atoms.append(new_atom)

    # add new bonds, angles, dihedrals to the molecule
    b = get_bond(param_top, 'c', new_atom.type)
    my_o_c_bond = Bond(new_atom, my_C_top, b.type, b._params)
    molecule.bonds.append(my_o_c_bond)

    new_angles = []
    for b in my_C_top.bonds:
        a1, a2 = b.atoms
        angle = get_angle(param_top, a1.type, a2.type, new_atom.type)
        my_angle = Angle(a1, a2, new_atom, angle.type, angle._params)
        molecule.angles.append(my_angle)
        new_angles.append(my_angle)

    new_dihedrals = []
    for ang in my_C_top.angles:
        a1, a2, a3 = ang.atoms
        my_types = [a.type for a in ang.atoms]

        if Counter(my_types) == Counter(['c', 'o', 'ca']):
            is_improper = True # this is an improper
        else:
            is_improper = False

        dih = get_dihedral(param_top, a1.type, a2.type, a3.type, new_atom.type, is_improper=is_improper)
        my_dihedral = Dihedral(a1, a2, a3, new_atom, dih.type, dih._params)
        molecule.dihedrals.append(my_dihedral)
        new_dihedrals.append(my_dihedral)

    new_atom.bonds.append(my_o_c_bond)
    new_atom.angles.extend(new_angles)
    new_atom.dihedrals.extend(new_dihedrals)

    # add them to C atom as well
    my_C_top.bonds.append(my_o_c_bond)
    my_C_top.angles.extend(new_angles)
    my_C_top.dihedrals.extend(new_dihedrals)

    print(f'Added O atom to broken amide C atom index {my_C.index} as atom index {total_atoms}')
    total_atoms += 1

# add H to the C on broken amide bonds
for n,my_C in enumerate(broken_amide_Cs):
    my_C_top = top.atoms[my_C.index]
    
    for b in my_C_top.bonds:
        has_oh = len([a for a in b.atoms if a.type == 'oh']) == 1
        if has_oh:
            my_O_top = [a for a in b.atoms if a.type == 'oh'][0]
        
    # add H if protonated
    if n < n_protonated:
        # pull most attributes from the parameterized fragment
        ref_atom = ho_atom
        atype = ref_atom.type
        atomname = ref_atom.atomname
        cgnr = ref_atom.cgnr
        charge = ref_atom.charge
        mass = ref_atom.mass
        atomtype = ref_atom.atomtype

        # pull other attributes from the current molecule
        num = my_C_top.molecule.n_atoms + 1
        resnum = my_C_top.resnum
        resname = my_C_top.resname
        molecule = my_C_top.molecule

        # create new atom and add to molecule
        new_H_atom_line = f'{num:7d} {atype:4s}\t\t{resnum} {resname:8s} {atomname:9s}\t{cgnr:d}\t{charge:.8f}\t{mass:.8f}\n'
        new_H_atom = Atom(total_atoms, new_H_atom_line)
        new_H_atom.atomtype = atomtype
        new_H_atom.molecule = molecule
        molecule.atoms.append(new_H_atom)

        # add new bonds, angles, dihedrals to the molecule
        b = get_bond(param_top, 'oh', new_H_atom.type)
        my_oh_ho_bond = Bond(my_O_top, new_H_atom, b.type, b._params)
        molecule.bonds.append(my_oh_ho_bond)

        for b in my_O_top.bonds:
            a1, a2 = b.atoms
            print(a1.type, a2.type)
            angle = get_angle(param_top, a1.type, a2.type, new_H_atom.type)
            my_angle = Angle(a1, a2, new_H_atom, angle.type, angle._params)
            molecule.angles.append(my_angle)

        for ang in my_O_top.angles:
            a1, a2, a3 = ang.atoms
            dih = get_dihedral(param_top, a1.type, a2.type, a3.type, new_H_atom.type)
            my_dihedral = Dihedral(a1, a2, a3, new_H_atom, dih.type, dih._params)
            molecule.dihedrals.append(my_dihedral)

        print(f'Added H atom to broken amide C atom index {my_C.index} as atom index {total_atoms}')
        total_atoms += 1

print('\nTotal atoms in topology:', total_atoms)

Added O atom to broken amide C atom index 12 as atom index 35657
Added O atom to broken amide C atom index 4105 as atom index 35658
Added O atom to broken amide C atom index 278 as atom index 35659
Added O atom to broken amide C atom index 7793 as atom index 35660
Added O atom to broken amide C atom index 8039 as atom index 35661
Added O atom to broken amide C atom index 7947 as atom index 35662
Added O atom to broken amide C atom index 4545 as atom index 35663
Added O atom to broken amide C atom index 8083 as atom index 35664
Added O atom to broken amide C atom index 8751 as atom index 35665
Added O atom to broken amide C atom index 9124 as atom index 35666
Added O atom to broken amide C atom index 4786 as atom index 35667
Added O atom to broken amide C atom index 4879 as atom index 35668
Added O atom to broken amide C atom index 7070 as atom index 35669
Added O atom to broken amide C atom index 10045 as atom index 35670
Added O atom to broken amide C atom index 12146 as atom index 35

In [11]:
top.write('./chlorinated.top')

Finished writing topology to ./chlorinated.top!


In [12]:
# reassign global atom indices to the GRO file
u = mda.Universe('added_atoms.gro')
global_map = {}
new_idx = 0

# start with polymer atoms
n_polymer_mols = len(top.molecules) - 4  # last 4 molecules are non-polymer
for i in range(n_polymer_mols):
    polymer_res = u.select_atoms(f'resname PA{i+1}')
    for atom in polymer_res:
        global_map[atom.ix] = new_idx
        new_idx += 1

print(f'Next available global atom index after polymer atoms: {new_idx}')

# then non-polymer atoms (first, waters)
for atom in u.select_atoms('resname SOL'):
    global_map[atom.ix] = new_idx
    new_idx += 1

print(f'Next available global atom index after water atoms: {new_idx}')

n_mols_NA_1 = top.molecules[-3].n_mols # NA molecules are split in topology, this is first set
for atom in u.select_atoms('resname NA')[:n_mols_NA_1]:
    global_map[atom.ix] = new_idx
    new_idx += 1

print(f'Next available global atom index after 1st Na atoms: {new_idx}')

# then Cl ions
n_mols_CL = top.molecules[-2].n_mols
for atom in u.select_atoms('resname CL'):
    global_map[atom.ix] = new_idx
    new_idx += 1

print(f'Next available global atom index after Cl atoms: {new_idx}')

for atom in u.select_atoms('resname NA')[n_mols_NA_1:]:
    global_map[atom.ix] = new_idx
    new_idx += 1

print(f'Next available global atom index after 2nd Na atoms: {new_idx}')

# build ids robustly and warn about any unmapped atoms
ids = []
missing = []
for atom in u.atoms:
    key = atom.ix
    if key in global_map:
        ids.append(global_map[key])
    else:
        ids.append(-1)             # or raise if you prefer
        missing.append((atom.ix, atom.name, atom.resname))

if missing:
    print(f'Warning: {len(missing)} unmapped atoms, examples: {missing[:5]}')

u.add_TopologyAttr('ids', [global_map[atom.ix] for atom in u.atoms])
u.atoms.sort('ids').write('chlorinated.gro')

Next available global atom index after polymer atoms: 18109
Next available global atom index after water atoms: 35851
Next available global atom index after 1st Na atoms: 35863
Next available global atom index after Cl atoms: 35875
Next available global atom index after 2nd Na atoms: 35940
